# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-funded users of PhysioNet.

## Import packages

In [ ]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

## Setup with Synthetic Data

In [ ]:
# Create synthetic person map
synthetic_lists = [list(range(100000000, 100000009)), list(range(1, 10))]
df_synthetic_person_map = pd.DataFrame(synthetic_lists).transpose()
df_synthetic_person_map.columns=['person_id', 'physionet_id']

In [ ]:
test_path = os.path.join("..", "tests", "data", "synthetic_names")

In [ ]:
# Get the synthetic names for PhysioNet and NIH
df_synthetic_physionet_users = get_physionet_users(os.path.join(test_path,'physionet_users.csv'), df_synthetic_person_map, first_name_as_initial=True)
df_synthetic_physionet_users.head()

In [ ]:
synthetic_investigators = get_investigators(pd.read_csv(os.path.join(test_path, "nih_investigators.csv")), first_name_as_initial=True)
synthetic_investigators[0:]

In [ ]:
synthetic_authors = get_authors(pd.read_csv(os.path.join(test_path, "nih_authors.csv")), first_name_as_initial=True)
synthetic_authors[0:]

## Link Synthetic Users

In [ ]:
limit = None
df_physionet_users = link_users(df_synthetic_physionet_users, synthetic_investigators, match_group="investigators", limit=limit)

In [ ]:
df_physionet_users = link_users(df_synthetic_physionet_users, synthetic_authors, match_group="authors", limit=limit)

In [ ]:
df_physionet_users.head(10)

## Setup with Actual Data

In [ ]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [ ]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
df_person_map = pd.read_csv(path)
df_person_map.head(3)

## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [ ]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
df_physionet_users = get_physionet_users(path, df_person_map, first_name_as_initial=True)
df_physionet_users = df_physionet_users[0:9]
df_physionet_users.head(3)

## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [ ]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
df_projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)
df_projects.head(3)

In [ ]:
# Get the names of Principal Investigators
investigators = get_investigators(df_projects, first_name_as_initial=True)
investigators[0:3]

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [ ]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
df_publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)
df_publications.head(3)

In [ ]:
# Get the names of authors
authors = get_authors(df_publications, first_name_as_initial=True)
authors[0:3]

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [ ]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
df_physionet_users = link_users(df_physionet_users, investigators, match_group="investigators", limit=limit)

In [ ]:
# Match NIH authors to PhysioNet users
df_physionet_users = link_users(df_physionet_users, authors, match_group="authors", limit=limit)

In [ ]:
df_physionet_users.head(5)

## Save the results

In [ ]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
df_physionet_users.to_csv(normalized_path, index=False)